In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.proxy import Proxy, ProxyType
from selenium.common.exceptions import ElementClickInterceptedException
import time
import os

In [ ]:
KONU_ID = 88
DOWNLOAD_PATH = 'files_gyh'
FILE_EXT = "xls" # or "csv"
LOCALE = "tr"
HOMEPAGE = f"https://biruni.tuik.gov.tr/medas/?kn={KONU_ID}&locale={LOCALE}"

In [ ]:
prefs = {
   "download.default_directory": DOWNLOAD_PATH,
   "savefile.default_directory": DOWNLOAD_PATH
}
options = webdriver.ChromeOptions()
options.add_experimental_option('prefs', prefs)
prxy = Proxy()
prxy.proxy_type = ProxyType.MANUAL
prxy.http_proxy = "proxy.address:8080"
prxy.ssl_proxy = "proxy.address:8080"
options.proxy = prxy
driver = webdriver.Chrome(options=options)

In [4]:
if not os.path.exists(DOWNLOAD_PATH):
    os.makedirs(DOWNLOAD_PATH)

In [5]:
def check_loading():
    time.sleep(1)
    loading = driver.find_elements(By.CLASS_NAME, "z-loading-indicator")
    while len(loading) > 0:
        time.sleep(1)
        loading = driver.find_elements(By.CLASS_NAME, "z-loading-indicator")

In [6]:
def download_file(itemtxt, dim):    
    download_btns = driver.find_elements(By.TAG_NAME, "img")
    valid_dl_btn = [btn for btn in download_btns if btn.get_attribute("src") and btn.get_attribute("src").endswith(f"{FILE_EXT}.png")][0]
    valid_dl_btn.click()
    time.sleep(4)
    
    if not os.path.exists(f"{DOWNLOAD_PATH}\\{dim}"):
        os.makedirs(f"{DOWNLOAD_PATH}\\{dim}")
    new_file_name = f"{DOWNLOAD_PATH}\\{dim}\\{itemtxt.strip()}.{FILE_EXT}"
    os.rename(f"{DOWNLOAD_PATH}\\pivot.{FILE_EXT}", new_file_name)

In [10]:
hierarchy = {}
driver.get(HOMEPAGE)
check_loading()

selects_dte = driver.find_elements(By.TAG_NAME, "select")
years = selects_dte[1].find_elements(By.TAG_NAME, "option")
year_texts = [year.text for year in years if year.text != 'Hepsi']

# en güncel baz yıl
year_texts = ["2009"]

for year in year_texts:
    driver.get(HOMEPAGE)
    check_loading()

    selects_dte = driver.find_elements(By.TAG_NAME, "select")
    year_opts = selects_dte[1].find_elements(By.TAG_NAME, "option")
    curr_year = [y for y in year_opts if y.text == year][0]
    curr_year.click()
    check_loading()

    litems = driver.find_elements(By.CLASS_NAME, "z-listitem")
    hierarchy[year] = {}
    for litem in litems:
        litem.click()
        check_loading()
        vboxes = driver.find_elements(By.CLASS_NAME, "z-vbox")
        dimensions = vboxes[0].find_elements(By.CLASS_NAME, "z-listitem")
        selectable_dims = [d.text for d in dimensions if not ((LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.'))]
        hierarchy[year][litem.text] = selectable_dims

        

In [ ]:
for year, rows in hierarchy.items():
    for row, dims in rows.items():
        for dim in dims:
            driver.get(HOMEPAGE)
            check_loading()
            
            selects_dte = driver.find_elements(By.TAG_NAME, "select")
            year_opts = selects_dte[1].find_elements(By.TAG_NAME, "option")
            curr_year = [y for y in year_opts if y.text == year][0]
            curr_year.click()
            check_loading()
            
            litems = driver.find_elements(By.CLASS_NAME, "z-listitem")
            curr_item = [litem for litem in litems if litem.text == row][0]
            curr_item.click()
            litem_text = curr_item.text
            check_loading()
            
            vboxes = driver.find_elements(By.CLASS_NAME, "z-vbox")
            dimensions = vboxes[0].find_elements(By.CLASS_NAME, "z-listitem")
            selected_dims = [d.text for d in dimensions if (LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.')]
            selectable_dims = [d for d in dimensions if not ((LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.'))]
            # selectable_dims[0].click()
            curr_dim = [d for d in selectable_dims if d.text == dim][0]
            selected_dims.append(curr_dim.text)
            curr_dim.click()
            check_loading()
            
            toolbar_btns = driver.find_elements(By.CLASS_NAME, "z-toolbarbutton")
            add_btn = [btn for btn in toolbar_btns if (LOCALE == 'tr' and btn.get_attribute("title") == "Göstergeleri Ekle") or (LOCALE == 'en' and btn.get_attribute("title") == 'Add Measurement(s)')]
            if len(add_btn) > 0 and (LOCALE == 'tr' and add_btn[0].text.strip() == "Göstergeleri Ekle") or (LOCALE == 'en' and add_btn[0].text.strip() == 'Add Measurement(s)'):
                add_btn[0].click()
                check_loading()
            
            vbox_btns = vboxes[0].find_elements(By.TAG_NAME, "button")
            ok_btn = [btn for btn in vbox_btns if (LOCALE == 'tr' and btn.text == "Tamam") or (LOCALE == 'en' and btn.text == 'Ok')][0]
            ok_btn.click()
            check_loading()
            
            captions = driver.find_elements(By.CLASS_NAME, "z-caption-content")
            dim_boxes = [cap for cap in captions if cap.text.strip() in selected_dims]
            for dim_box in dim_boxes:
                list_opts = dim_box.find_element(By.XPATH, "../../..").find_elements(By.CLASS_NAME, "z-listitem")
                if (LOCALE == 'tr' and list_opts[0].text == '<Hepsi>') or (LOCALE == 'en' and list_opts[0].text == '<All>'):
                    list_opts[0].click()
                    time.sleep(1)
            
            toolbar_btns = driver.find_elements(By.CLASS_NAME, "z-toolbarbutton")
            add_btn = [btn for btn in toolbar_btns if (LOCALE == 'tr' and btn.get_attribute("title") == "Göstergeleri Ekle") or (LOCALE == 'en' and btn.get_attribute("title") == 'Add Measurement(s)')][0]
            add_btn.click()
            check_loading()
            
            page_btns = driver.find_elements(By.TAG_NAME, "button")
            next_btn = [btn for btn in page_btns if (LOCALE == 'tr' and btn.text == "İleri") or (LOCALE == 'en' and btn.text == 'Forward')][0]
            next_btn.click()
            check_loading()
            
            selects = driver.find_elements(By.TAG_NAME, "select")
            options = selects[-1].find_elements(By.TAG_NAME, "option")
            options[-1].click()  #aylık
            check_loading()
            
            inputs = driver.find_elements(By.CLASS_NAME, "z-textbox")
            year_min = [inp for inp in inputs if inp.get_attribute("placeholder") == "min"][0]
            # year_min.send_keys("2005")
            year_min.send_keys(year)
            
            search_btns = driver.find_elements(By.TAG_NAME, "img")
            search_btn = [btn for btn in search_btns if btn.get_attribute("src") and btn.get_attribute("src").endswith("S6.png")][0]
            search_btn.click()
            
            time.sleep(2)
            rows_chbox = driver.find_elements(By.CLASS_NAME, "z-listheader-checkable")
            rows_chbox[0].click()
            check_loading()
            
            page2_btns = driver.find_elements(By.TAG_NAME, "button")
            next_btn2 = [btn for btn in page2_btns if (LOCALE == 'tr' and btn.text == "İleri") or (LOCALE == 'en' and btn.text == 'Forward')][0]
            next_btn2.click()
            check_loading()

            rows_chbox2 = driver.find_elements(By.CLASS_NAME, "z-listheader-checkable")
            rows_chbox2[1].click()
            check_loading()

            page3_btns = driver.find_elements(By.TAG_NAME, "button")
            report_btn = [btn for btn in page3_btns if (LOCALE == 'tr' and btn.text == "Rapor Oluştur") or (LOCALE == 'en' and btn.text == 'Create Report')][0]
            report_btn.click()
            check_loading()
            
            download_file(litem_text, dim)
